In [ ]:
# SMS Spam Detection - Exploratory Data Analysis

**Author**: Data Scientist  
**Date**: 15/06/2025  
**Task**: DS-001 - Comprehensive EDA  
**Dataset**: SMS Spam Collection (5,574 messages)

## Objectives
1. Load and validate dataset integrity
2. Analyze class distribution and imbalance
3. Explore text characteristics and patterns
4. Identify spam vs ham distinguishing features
5. Generate insights for preprocessing strategy

## Target Performance
- **Precision**: ≥92%
- **Recall**: ≥88% 
- **F1-Score**: ≥90%
- **Priority**: Minimize false positives (ham marked as spam)


In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re
from wordcloud import WordCloud
import warnings
warnings.filterwarnings('ignore')

# Set style for visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

# Set random seed for reproducibility
np.random.seed(42)

print("📚 Libraries imported successfully!")
print(f"📊 Analysis Date: 15/06/2025")


In [ ]:
## 1. Data Loading and Basic Validation


In [ ]:
# Load the dataset
try:
    df = pd.read_csv('../data/SMSSPamCollection', 
                     sep='\t', 
                     header=None, 
                     names=['label', 'message'],
                     encoding='utf-8')
    print("✅ Dataset loaded successfully!")
except Exception as e:
    print(f"❌ Error loading dataset: {e}")
    
# Basic dataset information
print(f"\n📊 Dataset Shape: {df.shape}")
print(f"📊 Columns: {list(df.columns)}")
print(f"📊 Memory Usage: {df.memory_usage(deep=True).sum() / 1024:.2f} KB")

# Display first few rows
print("\n📝 First 5 rows:")
print(df.head())

print("\n📊 Data Types:")
print(df.dtypes)


In [ ]:
# Data quality checks
print("🔍 DATA QUALITY ASSESSMENT")
print("=" * 40)

# Check for missing values
missing_values = df.isnull().sum()
print(f"📊 Missing Values:")
print(missing_values)

# Check for duplicates
duplicates = df.duplicated().sum()
print(f"\n📊 Duplicate Rows: {duplicates}")

if duplicates > 0:
    print("\n🔍 Sample duplicate messages:")
    duplicate_messages = df[df.duplicated(keep=False)].sort_values('message')
    print(duplicate_messages.head(10))

# Check unique labels
unique_labels = df['label'].unique()
print(f"\n📊 Unique Labels: {unique_labels}")

# Check for empty messages
empty_messages = df['message'].str.strip().str.len() == 0
print(f"📊 Empty Messages: {empty_messages.sum()}")

# Basic statistics
print(f"\n📊 Dataset Statistics:")
print(f"Total messages: {len(df):,}")
print(f"Average message length: {df['message'].str.len().mean():.1f} characters")
print(f"Median message length: {df['message'].str.len().median():.1f} characters")


In [ ]:
## 2. Class Distribution Analysis


In [ ]:
# Class distribution
class_counts = df['label'].value_counts()
class_percentages = df['label'].value_counts(normalize=True) * 100

print("📊 CLASS DISTRIBUTION ANALYSIS")
print("=" * 40)
print(f"Ham Messages: {class_counts['ham']:,} ({class_percentages['ham']:.2f}%)")
print(f"Spam Messages: {class_counts['spam']:,} ({class_percentages['spam']:.2f}%)")
print(f"Total Messages: {df.shape[0]:,}")

# Calculate imbalance ratio
imbalance_ratio = class_counts['ham'] / class_counts['spam']
print(f"\n⚠️ Imbalance Ratio (Ham:Spam): {imbalance_ratio:.1f}:1")

# Assess imbalance severity
if imbalance_ratio > 5:
    print("🚨 SEVERE CLASS IMBALANCE DETECTED!")
    print("   Recommendation: Use specialized techniques for imbalanced data")
elif imbalance_ratio > 2:
    print("⚠️ Moderate class imbalance detected")
    print("   Recommendation: Consider resampling or cost-sensitive learning")
else:
    print("✅ Class distribution is relatively balanced")

# Create visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Bar plot
class_counts.plot(kind='bar', ax=ax1, color=['skyblue', 'salmon'])
ax1.set_title('Message Count by Class', fontsize=14, fontweight='bold')
ax1.set_ylabel('Number of Messages')
ax1.set_xlabel('Class')
ax1.tick_params(axis='x', rotation=0)

# Add value labels on bars
for i, v in enumerate(class_counts.values):
    ax1.text(i, v + 50, f'{v:,}', ha='center', va='bottom', fontweight='bold')

# Pie chart
ax2.pie(class_counts.values, labels=class_counts.index, autopct='%1.1f%%', 
        colors=['skyblue', 'salmon'], startangle=90)
ax2.set_title('Class Distribution', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
## 3. Message Length Analysis


In [ ]:
# Add message length features
df['message_length'] = df['message'].str.len()
df['word_count'] = df['message'].str.split().str.len()
df['avg_word_length'] = df['message_length'] / df['word_count']

# Message length statistics by class
print("📊 MESSAGE LENGTH ANALYSIS")
print("=" * 50)

length_stats = df.groupby('label')['message_length'].agg(['count', 'mean', 'median', 'std', 'min', 'max']).round(2)
print("📏 Character Length Statistics by Class:")
print(length_stats)

print("\n📝 Word Count Statistics by Class:")
word_stats = df.groupby('label')['word_count'].agg(['count', 'mean', 'median', 'std', 'min', 'max']).round(2)
print(word_stats)

print("\n📊 Average Word Length by Class:")
avg_word_stats = df.groupby('label')['avg_word_length'].agg(['mean', 'median', 'std']).round(2)
print(avg_word_stats)

# Statistical significance test
from scipy import stats
spam_lengths = df[df['label'] == 'spam']['message_length']
ham_lengths = df[df['label'] == 'ham']['message_length']

# Perform t-test
t_stat, p_value = stats.ttest_ind(spam_lengths, ham_lengths)
print(f"\n🧪 T-test for length difference:")
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.6f}")
if p_value < 0.05:
    print("✅ Significant difference in message lengths between classes")
else:
    print("❌ No significant difference in message lengths")


In [ ]:
# Visualize message length distributions
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Character length distribution
axes[0,0].hist(df[df['label'] == 'ham']['message_length'], bins=50, alpha=0.7, 
               label='Ham', color='skyblue', density=True)
axes[0,0].hist(df[df['label'] == 'spam']['message_length'], bins=50, alpha=0.7, 
               label='Spam', color='salmon', density=True)
axes[0,0].set_title('Message Length Distribution (Characters)', fontweight='bold')
axes[0,0].set_xlabel('Number of Characters')
axes[0,0].set_ylabel('Density')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# Word count distribution
axes[0,1].hist(df[df['label'] == 'ham']['word_count'], bins=50, alpha=0.7, 
               label='Ham', color='skyblue', density=True)
axes[0,1].hist(df[df['label'] == 'spam']['word_count'], bins=50, alpha=0.7, 
               label='Spam', color='salmon', density=True)
axes[0,1].set_title('Word Count Distribution', fontweight='bold')
axes[0,1].set_xlabel('Number of Words')
axes[0,1].set_ylabel('Density')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

# Box plots for better comparison
df.boxplot(column='message_length', by='label', ax=axes[1,0])
axes[1,0].set_title('Message Length by Class (Box Plot)', fontweight='bold')
axes[1,0].set_xlabel('Class')
axes[1,0].set_ylabel('Character Count')

df.boxplot(column='word_count', by='label', ax=axes[1,1])
axes[1,1].set_title('Word Count by Class (Box Plot)', fontweight='bold')
axes[1,1].set_xlabel('Class')
axes[1,1].set_ylabel('Word Count')

plt.tight_layout()
plt.show()

# Length percentiles for insights
print("\n📊 MESSAGE LENGTH PERCENTILES")
print("=" * 40)
for label in ['ham', 'spam']:
    subset = df[df['label'] == label]['message_length']
    print(f"\n{label.upper()} Messages:")
    print(f"  25th percentile: {subset.quantile(0.25):.0f} chars")
    print(f"  50th percentile: {subset.quantile(0.50):.0f} chars") 
    print(f"  75th percentile: {subset.quantile(0.75):.0f} chars")
    print(f"  95th percentile: {subset.quantile(0.95):.0f} chars")


In [ ]:
## 4. Character-Level Pattern Analysis


In [ ]:
# Character-level feature extraction
def extract_char_features(text):
    """Extract character-level features from text"""
    if pd.isna(text) or len(text) == 0:
        return {
            'punct_count': 0, 'digit_count': 0, 'upper_count': 0,
            'punct_ratio': 0, 'digit_ratio': 0, 'upper_ratio': 0
        }
    
    punct_count = sum(1 for c in text if c in '!@#$%^&*()_+-=[]{}|;:,.<>?')
    digit_count = sum(1 for c in text if c.isdigit())
    upper_count = sum(1 for c in text if c.isupper())
    
    return {
        'punct_count': punct_count,
        'digit_count': digit_count,
        'upper_count': upper_count,
        'punct_ratio': punct_count / len(text) if len(text) > 0 else 0,
        'digit_ratio': digit_count / len(text) if len(text) > 0 else 0,
        'upper_ratio': upper_count / len(text) if len(text) > 0 else 0
    }

# Apply feature extraction
char_features = df['message'].apply(extract_char_features)
char_df = pd.DataFrame(char_features.tolist())
df = pd.concat([df, char_df], axis=1)

# Analyze character patterns by class
print("📊 CHARACTER-LEVEL PATTERN ANALYSIS")
print("=" * 50)

char_stats = df.groupby('label')[['punct_ratio', 'digit_ratio', 'upper_ratio']].agg(['mean', 'median', 'std']).round(4)
print("📝 Character Pattern Statistics by Class:")
print(char_stats)

# Statistical tests for character features
print("\n🧪 Statistical Significance Tests:")
for feature in ['punct_ratio', 'digit_ratio', 'upper_ratio']:
    spam_values = df[df['label'] == 'spam'][feature]
    ham_values = df[df['label'] == 'ham'][feature]
    
    t_stat, p_value = stats.ttest_ind(spam_values, ham_values)
    print(f"{feature.replace('_', ' ').title()}:")
    print(f"  Spam mean: {spam_values.mean():.4f}, Ham mean: {ham_values.mean():.4f}")
    print(f"  T-statistic: {t_stat:.4f}, P-value: {p_value:.6f}")
    if p_value < 0.05:
        print("  ✅ Significant difference")
    else:
        print("  ❌ No significant difference")
    print()

# Special character analysis
print("🔍 SPECIAL PATTERN DETECTION")
print("=" * 40)

# URLs, phone numbers, money mentions
df['has_url'] = df['message'].str.contains(r'http[s]?://|www\.', case=False, na=False)
df['has_phone'] = df['message'].str.contains(r'\b\d{10,11}\b|\b\d{3}[-.]?\d{3}[-.]?\d{4}\b', na=False)
df['has_money'] = df['message'].str.contains(r'£|\$|money|cash|prize|win|free|offer', case=False, na=False)
df['has_urgency'] = df['message'].str.contains(r'urgent|act now|limited time|expires|claim|call now', case=False, na=False)

special_patterns = ['has_url', 'has_phone', 'has_money', 'has_urgency']
for pattern in special_patterns:
    spam_rate = df[df['label'] == 'spam'][pattern].mean()
    ham_rate = df[df['label'] == 'ham'][pattern].mean()
    
    print(f"{pattern.replace('_', ' ').replace('has ', '').title()} Mentions:")
    print(f"  Spam: {spam_rate:.1%} | Ham: {ham_rate:.1%} | Ratio: {spam_rate/max(ham_rate, 0.001):.1f}x")
    print()


In [ ]:
## 5. Word Frequency Analysis


In [ ]:
# Word frequency analysis
import string
from collections import Counter

def clean_text_for_analysis(text):
    """Clean text for word frequency analysis"""
    if pd.isna(text):
        return ""
    # Convert to lowercase and remove punctuation
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text

# Prepare text data
df['clean_message'] = df['message'].apply(clean_text_for_analysis)

# Get word frequencies for each class
ham_text = ' '.join(df[df['label'] == 'ham']['clean_message'])
spam_text = ' '.join(df[df['label'] == 'spam']['clean_message'])

ham_words = ham_text.split()
spam_words = spam_text.split()

# Remove common stop words (basic list)
stop_words = set(['the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by', 
                  'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'do', 'does', 'did',
                  'will', 'would', 'could', 'should', 'may', 'might', 'can', 'shall', 'must', 'i', 'you', 'he',
                  'she', 'it', 'we', 'they', 'me', 'him', 'her', 'us', 'them', 'my', 'your', 'his', 'its', 'our'])

ham_words_filtered = [word for word in ham_words if word not in stop_words and len(word) > 2]
spam_words_filtered = [word for word in spam_words if word not in stop_words and len(word) > 2]

# Get top words
ham_word_freq = Counter(ham_words_filtered)
spam_word_freq = Counter(spam_words_filtered)

print("📊 WORD FREQUENCY ANALYSIS") 
print("=" * 50)

print(f"📝 Total unique words:")
print(f"  Ham: {len(ham_word_freq):,} unique words")
print(f"  Spam: {len(spam_word_freq):,} unique words")

print(f"\n📊 Top 15 words in HAM messages:")
for word, count in ham_word_freq.most_common(15):
    print(f"  {word}: {count:,}")

print(f"\n📊 Top 15 words in SPAM messages:")
for word, count in spam_word_freq.most_common(15):
    print(f"  {word}: {count:,}")

# Find words that appear much more in spam vs ham
spam_specific_words = []
ham_specific_words = []

for word in spam_word_freq:
    spam_count = spam_word_freq[word]
    ham_count = ham_word_freq.get(word, 0)
    
    if spam_count >= 5:  # Only consider words that appear at least 5 times
        spam_ratio = spam_count / max(ham_count, 1)
        if spam_ratio >= 3:  # Words that appear 3x more in spam
            spam_specific_words.append((word, spam_count, ham_count, spam_ratio))

spam_specific_words.sort(key=lambda x: x[3], reverse=True)

print(f"\n🚨 Words strongly associated with SPAM (top 10):")
for word, spam_count, ham_count, ratio in spam_specific_words[:10]:
    print(f"  {word}: {spam_count} spam, {ham_count} ham (ratio: {ratio:.1f}x)")

# Calculate vocabulary richness
ham_vocab_richness = len(ham_word_freq) / len(ham_words_filtered) if ham_words_filtered else 0
spam_vocab_richness = len(spam_word_freq) / len(spam_words_filtered) if spam_words_filtered else 0

print(f"\n📈 Vocabulary Richness (unique words / total words):")
print(f"  Ham: {ham_vocab_richness:.4f}")
print(f"  Spam: {spam_vocab_richness:.4f}")


In [ ]:
# Create word clouds
try:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))
    
    # Ham word cloud
    ham_wordcloud = WordCloud(width=800, height=400, 
                              background_color='white',
                              colormap='Blues',
                              max_words=100).generate(ham_text)
    
    ax1.imshow(ham_wordcloud, interpolation='bilinear')
    ax1.set_title('Most Common Words in HAM Messages', fontsize=16, fontweight='bold')
    ax1.axis('off')
    
    # Spam word cloud
    spam_wordcloud = WordCloud(width=800, height=400, 
                               background_color='white',
                               colormap='Reds',
                               max_words=100).generate(spam_text)
    
    ax2.imshow(spam_wordcloud, interpolation='bilinear')
    ax2.set_title('Most Common Words in SPAM Messages', fontsize=16, fontweight='bold')
    ax2.axis('off')
    
    plt.tight_layout()
    plt.show()
    
except ImportError:
    print("⚠️ WordCloud not available. Showing text-based analysis instead.")
    
# Create comparative bar charts of top words
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

# Top ham words
ham_top_words = dict(ham_word_freq.most_common(15))
ax1.barh(list(ham_top_words.keys()), list(ham_top_words.values()), color='skyblue')
ax1.set_title('Top 15 Words in HAM Messages', fontweight='bold')
ax1.set_xlabel('Frequency')
ax1.invert_yaxis()

# Top spam words
spam_top_words = dict(spam_word_freq.most_common(15))
ax2.barh(list(spam_top_words.keys()), list(spam_top_words.values()), color='salmon')
ax2.set_title('Top 15 Words in SPAM Messages', fontweight='bold')
ax2.set_xlabel('Frequency')
ax2.invert_yaxis()

plt.tight_layout()
plt.show()


In [ ]:
## 6. Sample Message Analysis


In [ ]:
# Analyze sample messages
print("📝 SAMPLE MESSAGE ANALYSIS")
print("=" * 50)

# Get representative samples
spam_samples = df[df['label'] == 'spam'].sample(n=5, random_state=42)
ham_samples = df[df['label'] == 'ham'].sample(n=5, random_state=42)

print("🚨 SPAM SAMPLES:")
print("-" * 30)
for i, (idx, row) in enumerate(spam_samples.iterrows(), 1):
    print(f"{i}. [{len(row['message'])} chars] {row['message'][:100]}...")
    print(f"   Features: {row['word_count']} words, {row['punct_ratio']:.3f} punct ratio, "
          f"URL: {row['has_url']}, Money: {row['has_money']}, Urgency: {row['has_urgency']}")
    print()

print("✅ HAM SAMPLES:")
print("-" * 30)
for i, (idx, row) in enumerate(ham_samples.iterrows(), 1):
    print(f"{i}. [{len(row['message'])} chars] {row['message'][:100]}...")
    print(f"   Features: {row['word_count']} words, {row['punct_ratio']:.3f} punct ratio, "
          f"URL: {row['has_url']}, Money: {row['has_money']}, Urgency: {row['has_urgency']}")
    print()

# Identify extreme cases
print("🔍 EXTREME CASES ANALYSIS")
print("=" * 40)

# Longest and shortest messages
longest_spam = df[df['label'] == 'spam'].nlargest(3, 'message_length')
shortest_spam = df[df['label'] == 'spam'].nsmallest(3, 'message_length')
longest_ham = df[df['label'] == 'ham'].nlargest(3, 'message_length')

print("📏 Longest SPAM messages:")
for i, (idx, row) in enumerate(longest_spam.iterrows(), 1):
    print(f"{i}. [{len(row['message'])} chars] {row['message'][:150]}...")
    print()

print("📏 Shortest SPAM messages:")
for i, (idx, row) in enumerate(shortest_spam.iterrows(), 1):
    print(f"{i}. [{len(row['message'])} chars] {row['message']}")
    print()

print("📏 Longest HAM messages:")
for i, (idx, row) in enumerate(longest_ham.iterrows(), 1):
    print(f"{i}. [{len(row['message'])} chars] {row['message'][:150]}...")
    print()


In [ ]:
## 7. Key Insights and Recommendations


In [ ]:
# Comprehensive insights summary
print("🎯 KEY INSIGHTS FROM EXPLORATORY DATA ANALYSIS")
print("=" * 60)

print("📊 DATASET CHARACTERISTICS:")
print(f"• Total messages: {len(df):,}")
print(f"• Class distribution: {df['label'].value_counts()['ham']:,} ham ({df['label'].value_counts(normalize=True)['ham']:.1%}), " 
      f"{df['label'].value_counts()['spam']:,} spam ({df['label'].value_counts(normalize=True)['spam']:.1%})")
print(f"• Imbalance ratio: {df['label'].value_counts()['ham'] / df['label'].value_counts()['spam']:.1f}:1 (SEVERE)")
print(f"• Data quality: No missing values, {df.duplicated().sum()} duplicates")

print("\n📏 MESSAGE LENGTH PATTERNS:")
ham_avg_len = df[df['label'] == 'ham']['message_length'].mean()
spam_avg_len = df[df['label'] == 'spam']['message_length'].mean()
print(f"• Ham messages: {ham_avg_len:.1f} chars average ({df[df['label'] == 'ham']['word_count'].mean():.1f} words)")
print(f"• Spam messages: {spam_avg_len:.1f} chars average ({df[df['label'] == 'spam']['word_count'].mean():.1f} words)")
print(f"• Length difference: {'Spam longer' if spam_avg_len > ham_avg_len else 'Ham longer'} by {abs(spam_avg_len - ham_avg_len):.1f} chars")

print("\n🔍 DISCRIMINATIVE FEATURES IDENTIFIED:")
print("• Spam messages have significantly higher rates of:")
for pattern in ['has_money', 'has_urgency', 'has_phone', 'has_url']:
    spam_rate = df[df['label'] == 'spam'][pattern].mean()
    ham_rate = df[df['label'] == 'ham'][pattern].mean()
    if spam_rate > ham_rate * 2:  # Only show patterns with strong difference
        print(f"  - {pattern.replace('has_', '').replace('_', ' ').title()}: {spam_rate:.1%} vs {ham_rate:.1%} ({spam_rate/max(ham_rate, 0.001):.1f}x higher)")

print("\n📝 VOCABULARY INSIGHTS:")
print(f"• Ham vocabulary richness: {len(ham_word_freq) / len(ham_words_filtered):.4f}")
print(f"• Spam vocabulary richness: {len(spam_word_freq) / len(spam_words_filtered):.4f}")
print("• Top spam-specific words reveal patterns:")
if spam_specific_words:
    for word, spam_count, ham_count, ratio in spam_specific_words[:5]:
        print(f"  - '{word}': {ratio:.1f}x more frequent in spam")

print("\n🎯 RECOMMENDATIONS FOR MODEL DEVELOPMENT:")
print("1. CLASS IMBALANCE HANDLING:")
print("   • Use stratified sampling for train/validation splits")
print("   • Consider SMOTE, class weighting, or cost-sensitive learning")
print("   • Focus on precision-recall metrics over accuracy")

print("\n2. FEATURE ENGINEERING PRIORITIES:")
print("   • Text length features (characters, words, sentences)")
print("   • Character-level patterns (punctuation, digits, uppercase ratios)")
print("   • Domain-specific patterns (URLs, phone numbers, money terms)")
print("   • Urgency language detection")
print("   • TF-IDF with n-grams (1-3) for vocabulary patterns")

print("\n3. MODEL SELECTION STRATEGY:")
print("   • Start with Naive Bayes (handles text well, good with imbalanced data)")
print("   • Try ensemble methods (Random Forest, XGBoost) with class weights")
print("   • Consider SVM with appropriate kernels")
print("   • Explore neural approaches if computational budget allows")

print("\n4. EVALUATION APPROACH:")
print("   • Primary metrics: Precision ≥92%, Recall ≥88%, F1-Score ≥90%")
print("   • Use stratified k-fold cross-validation")
print("   • Analyze confusion matrix for error patterns")
print("   • Test robustness against spam evasion techniques")

print("\n5. POTENTIAL CHALLENGES:")
print("   • Limited training data (5,572 messages)")
print("   • Severe class imbalance requires specialized handling")
print("   • Need to balance false positive vs false negative rates")
print("   • Model interpretability important for business acceptance")

print("\n✅ READY FOR NEXT PHASE:")
print("• Dataset fully characterized and validated")
print("• Key discriminative patterns identified")
print("• Feature engineering strategy defined") 
print("• Model development roadmap established")
print("• Success metrics and evaluation plan confirmed")

print("\n📋 NEXT STEPS (DS-002):")
print("• Implement text preprocessing pipeline")
print("• Create feature engineering functions")
print("• Set up stratified data splitting")
print("• Begin baseline model development")
